In [5]:

pip install pandas torch transformers datasets sacrebleu accelerate sentencepiece


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 5.7 MB/s eta 0:00:01
     |████████████████████████████████| 73.6 MB 34.0 MB/s eta 0:00:01
     |████████████████████████████████| 12.0 MB 18.5 MB/s eta 0:00:01
     |████████████████████████████████| 511 kB 27.3 MB/s eta 0:00:01
     |████████████████████████████████| 104 kB 29.0 MB/s eta 0:00:01
     |████████████████████████████████| 374 kB 26.5 MB/s eta 0:00:01
     |████████████████████████████████| 1.3 MB 33.4 MB/s eta 0:00:01
     |████████████████████████████████| 509 kB 84.8 MB/s eta 0:00:01
     |████████████████████████████████| 347 kB 25.2 MB/s eta 0:00:01
     |████████████████████████████████| 6.3 MB 41.2 MB/s eta 0:00:01
     |████████████████████████████████| 134 kB 28.3 MB/s eta 0:00:01
     |████████████████████████████████| 200 kB 21.9 MB/s eta 0:00:01
     |████████████████████████████████| 1.6 MB 31.8 MB/s eta 0:00:01
     |█████████████████

In [7]:
pip install protobuf

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 427 kB 3.7 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset
import sacrebleu

In [9]:
MODEL_NAME = "google/mt5-small"

# Shorter sequences to save memory
MAX_LENGTH = 64

# Smaller batch size on-device + gradient accumulation
BATCH_SIZE = 4          # what actually sits in memory at once
GRAD_ACCUM_STEPS = 4    # 4 * 4 = effective batch of 16

# Hinglish needs more training
HINGLISH_EPOCHS = 10
HINGLISH_LR = 5e-5

# Spanglish doing okay
SPANGLISH_EPOCHS = 8
SPANGLISH_LR = 3e-5

# Force CPU everywhere to avoid MPS OOM
FORCE_CPU = True

# Just in case: clear any old MPS cache (harmless on CPU)
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

In [10]:
print("Loading data...")
hing_train = pd.read_csv("data/hinglish_train.csv")
hing_val = pd.read_csv("data/hinglish_val.csv")
hing_test = pd.read_csv("data/hinglish_test.csv")

span_train = pd.read_csv("data/spanglish_train.csv")
span_val = pd.read_csv("data/spanglish_val.csv")
span_test = pd.read_csv("data/spanglish_test.csv")

print(f"Hinglish: {len(hing_train)} train, {len(hing_val)} val, {len(hing_test)} test")
print(f"Spanglish: {len(span_train)} train, {len(span_val)} val, {len(span_test)} test")

Loading data...
Hinglish: 743 train, 93 val, 93 test
Spanglish: 844 train, 105 val, 106 test


In [11]:
def tokenize_data(df, tokenizer, lang_name):
    """Convert CSV to tokenized dataset."""
    sources = [
        f"translate {lang_name} to english: {text}"
        for text in df["source"].astype(str)
    ]
    targets = df["target"].astype(str).tolist()

    inputs = tokenizer(
        sources, max_length=MAX_LENGTH, truncation=True, padding=False
    )
    labels = tokenizer(
        targets, max_length=MAX_LENGTH, truncation=True, padding=False
    )

    return Dataset.from_dict(
        {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": labels["input_ids"],
        }
    )

def compute_metrics(eval_pred, tokenizer):
    """Calculate BLEU and chrF."""
    predictions, labels = eval_pred

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in labels (ignore index) before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels]).score
    chrf = sacrebleu.corpus_chrf(decoded_preds, [decoded_labels]).score

    return {"bleu": bleu, "chrf": chrf}


def get_device():
    """Return device for evaluation."""
    if not FORCE_CPU and torch.cuda.is_available():
        return "cuda"
    # We deliberately ignore MPS to avoid OOMs
    return "cpu"

In [12]:
print("\n" + "=" * 80)
print("TRAINING HINGLISH MODEL (CPU)")
print("=" * 80)

# Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Enable gradient checkpointing to reduce memory (still helpful on CPU)
model.gradient_checkpointing_enable()

# Prepare data
train_dataset = tokenize_data(hing_train, tokenizer, "hinglish")
val_dataset = tokenize_data(hing_val, tokenizer, "hinglish")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Training settings
training_args = Seq2SeqTrainingArguments(
    output_dir="models/hinglish",
    num_train_epochs=HINGLISH_EPOCHS,
    learning_rate=HINGLISH_LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",          # alias for evaluation_strategy
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    generation_num_beams=4,
    fp16=False,                     # no mixed precision on CPU
    report_to="none",
    no_cuda=True,                   # disable CUDA
    use_mps_device=False,           # disable MPS explicitly
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda x: compute_metrics(x, tokenizer),
)

print("Training Hinglish...")
trainer.train()
trainer.save_model("models/hinglish/best_model")
tokenizer.save_pretrained("models/hinglish/best_model")
print("Hinglish model saved!")



TRAINING HINGLISH MODEL (CPU)


/Users/chidipothusiritha/Library/Python/3.9/lib/python/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
/Users/chidipothusiritha/Library/Python/3.9/lib/python/site-packages/transformers/training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
/var/folders/yq/pgkldf4s0ng9v2h3wf5m0chw0000gn/T/ipykernel_44033/4159004226.py:43: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trai

Training Hinglish...


Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,No log,21.686399,0.051170,2.170701
2,26.284300,19.516455,0.055686,2.194033


OverflowError: out of range integral type conversion attempted

In [ ]:
# =============================================================================
# TRAIN SPANGLISH
# =============================================================================

print("\n" + "=" * 80)
print("TRAINING SPANGLISH MODEL")
print("=" * 80)

# Load fresh tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.gradient_checkpointing_enable()

# Prepare data
train_dataset = tokenize_data(span_train, tokenizer, "spanglish")
val_dataset = tokenize_data(span_val, tokenizer, "spanglish")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Training settings (different LR and epochs)
training_args = Seq2SeqTrainingArguments(
    output_dir="models/spanglish",
    num_train_epochs=SPANGLISH_EPOCHS,
    learning_rate=SPANGLISH_LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    generation_num_beams=4,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda x: compute_metrics(x, tokenizer),
)

print("Training Spanglish...")
trainer.train()
trainer.save_model("models/spanglish/best_model")
tokenizer.save_pretrained("models/spanglish/best_model")
print("Spanglish model saved!")


In [ ]:
print("\n" + "=" * 80)
print("EVALUATION ON TEST SETS")
print("=" * 80)


def evaluate(model_path, test_df, lang_name):
    """Quick evaluation function."""
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

    device = get_device()
    model.to(device)
    model.eval()

    sources = [
        f"translate {lang_name} to english: {text}"
        for text in test_df["source"].astype(str)
    ]
    references = test_df["target"].astype(str).tolist()
    predictions = []

    # Generate predictions
    batch_size = 32
    for i in range(0, len(sources), batch_size):
        batch = sources[i : i + batch_size]
        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            max_length=MAX_LENGTH,
            truncation=True,
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model.generate(
                **encoded, max_length=MAX_LENGTH, num_beams=4
            )

        batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend([p.strip() for p in batch_preds])

    # Calculate metrics
    bleu = sacrebleu.corpus_bleu(predictions, [references]).score
    chrf = sacrebleu.corpus_chrf(predictions, [references]).score
    em = (
        100.0
        * sum(
            p.lower().strip() == r.lower().strip()
            for p, r in zip(predictions, references)
        )
        / len(references)
    )

    return predictions, bleu, chrf, em


# Evaluate Hinglish
print("\nEvaluating Hinglish...")
hing_preds, hing_bleu, hing_chrf, hing_em = evaluate(
    "models/hinglish/best_model", hing_test, "hinglish"
)

# Evaluate Spanglish
print("Evaluating Spanglish...")
span_preds, span_bleu, span_chrf, span_em = evaluate(
    "models/spanglish/best_model", span_test, "spanglish"
)

# Print results
print("\n" + "=" * 80)
print("FINAL RESULTS")
print("=" * 80)
print(f"{'Language':<12} {'BLEU':<10} {'chrF':<10} {'ExactMatch':<10}")
print("-" * 80)
print(f"{'Hinglish':<12} {hing_bleu:<10.2f} {hing_chrf:<10.2f} {hing_em:<10.2f}")
print(f"{'Spanglish':<12} {span_bleu:<10.2f} {span_chrf:<10.2f} {span_em:<10.2f}")
print("=" * 80)

# Save predictions
hing_test["mt5_prediction"] = hing_preds
span_test["mt5_prediction"] = span_preds
hing_test.to_csv("hinglish_mt5_results.csv", index=False)
span_test.to_csv("spanglish_mt5_results.csv", index=False)

print("\nDone! Predictions saved to CSV files.")
print("Models saved in models/hinglish/ and models/spanglish/")